In [1]:
%pip install simfile pandas

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import glob

# packs from my setup
songs_path = '/media/win/Users/Lenovo/AppData/Roaming/ITGmania/Songs/'
def my_setup(pack_name):
    return os.path.join(songs_path, pack_name)

def my_setup_srpg(which):
    return [
        my_setup(f'Stamina RPG {which}{suffix}')
        for suffix in ['', ' - SN', ' - FE']
    ]

def stash_path(pack_prefix):
    return glob.glob('./packs/' + pack_prefix + '*')

def stash_path_srpg(which):
    return stash_path('Stamina RPG %d' % (which,))

# chronological order
pack_paths = [
    *stash_path_srpg(1),
    *stash_path_srpg(2),
    *stash_path_srpg(3),
    *stash_path_srpg(4),
    *stash_path_srpg(5),
    *stash_path('Stamina RPG 6'),
    my_setup('Stamina RPG 6'),
    #my_setup('SRPG6 NNP'),
    *stash_path('ECS11'),
    *my_setup_srpg(7),
    *stash_path('Stamina RPG 7 Unlocks'),
    *stash_path('ECS12'),
    *my_setup_srpg(8),
    *stash_path('Stamina RPG 8 Unlocks'),
    *stash_path('ECS13'),
    *my_setup_srpg(9),
    *stash_path('Stamina RPG 9 Unlocks'),
    *stash_path('ECS14'),
    *my_setup_srpg(10),
]

release_order = {
    os.path.basename(path): i for i, path in enumerate(pack_paths)
}
pack_paths

['./packs/Stamina RPG 1 Unlocks',
 './packs/Stamina RPG 1',
 './packs/Stamina RPG 2 Unlocks',
 './packs/Stamina RPG 2',
 './packs/Stamina RPG 3 Unlocks',
 './packs/Stamina RPG 3',
 './packs/Stamina RPG 4 DLC',
 './packs/Stamina RPG 4 Unlocks',
 './packs/Stamina RPG 4',
 './packs/Stamina RPG 5 Unlocks',
 './packs/Stamina RPG 5',
 './packs/Stamina RPG 6',
 './packs/Stamina RPG 6 Unlocks',
 '/media/win/Users/Lenovo/AppData/Roaming/ITGmania/Songs/Stamina RPG 6',
 './packs/ECS11 - Lower',
 './packs/ECS11 - Mid',
 './packs/ECS11 - Upper',
 './packs/ECS11 - Upper Marathon',
 './packs/ECS11 - Lower Marathon',
 './packs/ECS11 - Speed',
 './packs/ECS11 - Mid Marathon',
 '/media/win/Users/Lenovo/AppData/Roaming/ITGmania/Songs/Stamina RPG 7',
 '/media/win/Users/Lenovo/AppData/Roaming/ITGmania/Songs/Stamina RPG 7 - SN',
 '/media/win/Users/Lenovo/AppData/Roaming/ITGmania/Songs/Stamina RPG 7 - FE',
 './packs/Stamina RPG 7 Unlocks',
 './packs/ECS12 - Mid',
 './packs/ECS12 - Mid Marathon',
 './packs/EC

In [2]:
import pandas as pd

try:
    existing_df = pd.read_json('../public/dataset.json')
    hash_to_data = {
        h: row
        for h, row in zip(
            existing_df['hash'], existing_df.to_numpy().tolist()
        )
    }
except FileNotFoundError:
    existing_df = None
    hash_to_data = {}

In [3]:
existing_df = None
hash_to_data = {}

In [4]:
import simfile
from simfile.dir import SimfilePack
from simfile.types import Simfile
from analysis.analyzer import SongAnalyzer
from analysis.hash import get_hash

def add_song_data(pack: SimfilePack | None, song: Simfile):
    pack_name = pack.name if pack else ''
    title = song.title
    if song.subtitle:
        title += ' ' + song.subtitle

    song_analyzer = SongAnalyzer(song)
    for chart_analyzer in song_analyzer.chart_analyzers.values():
        chart = chart_analyzer.chart
        if (chart.stepstype or '').lower() != 'dance-single':
            continue
        diff = (chart.difficulty or '').lower()
        chart_title = title
        if diff != 'challenge':
            if diff == 'beginner':
                diff = 'novice'
            chart_title += f' ({diff.title()})'
        
        try:
            meter = int(chart_analyzer.chart.meter)
        except ValueError:
            print(chart_title, ': invalid meter')
            continue

        chart_hash = get_hash(song, chart_analyzer.chart)
        if chart_hash in hash_to_data:
            existing_entry = hash_to_data[chart_hash]
            prev_pack_name = existing_entry[1]
            prev_order = release_order.get(prev_pack_name, -1)
            cur_order = release_order.get(pack_name, -1)
            prev_meter = existing_entry[3]

            if prev_meter != meter and prev_order < cur_order:
                # replace with newer rerate
                existing_entry[1] = pack_name
                existing_entry[3] = meter
                print(f'rerated {pack_name}/{chart_title} from {prev_meter} ({prev_pack_name}) to {meter}')
                continue
            elif prev_meter == meter and cur_order < prev_order:
                # replace with older pack
                existing_entry[1] = pack_name
                existing_entry[3] = meter
                print(f'reassigned {pack_name}/{chart_title} from pack {prev_pack_name}')
                continue
            else:
                print(f'skipping {pack_name}/{chart_title} (dupe hash)')
                continue
        
        print(f'analyzing {pack_name}/{chart_title}')

        stream_info = chart_analyzer.get_stream_info()
        total_stream = stream_info['total_stream']
        total_break = stream_info['total_break']
        bpms = stream_info['bpms']
        avg_bpm = stream_info['avg_bpm']

        if total_stream == 0:
            print(chart_title, ': no stream')
        else:
            mult = stream_info['quant'] / 16
            segments = stream_info['segments']
            hash_to_data[chart_hash] = [
                chart_title,
                pack_name,
                chart_hash,
                meter,
                round(avg_bpm * mult, 3),
                total_stream * mult,
                total_stream / (total_stream + total_break),
                round(bpms[0] * mult, 3),
                round(bpms[1] * mult, 3),
                segments,
                chart_analyzer.get_density_graph()
            ]

for pack_path in pack_paths:
    pack = SimfilePack(pack_path)
    for song in pack.simfiles(strict=False):
        add_song_data(pack, song)

# for sim_path in glob.glob('./packs/_misc/*.sm'):
#     song = simfile.open(sim_path, strict=False)
#     add_song_data(None, song)

/home/som1sezhi/Documents/projs/itg-scatter-plot/notebooks/.venv/lib/python3.10/site-packages/fs/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore


analyzing Stamina RPG 1 Unlocks/[21] [140] 140 BPM Stamina Killer
analyzing Stamina RPG 1 Unlocks/[20] [174] Twostament of Ian
analyzing Stamina RPG 1 Unlocks/[26] [250] Chambers of Shaolin
analyzing Stamina RPG 1 Unlocks/[21] [160] Eternal Nectar
analyzing Stamina RPG 1 Unlocks/[18] [174] Enjoy the Flight
analyzing Stamina RPG 1 Unlocks/[23] [249] Shit Pallette 2.0
analyzing Stamina RPG 1 Unlocks/[19] [173] Testament of Ian
analyzing Stamina RPG 1 Unlocks/[24] [255] Haughs o' Cromdale
analyzing Stamina RPG 1 Unlocks/[19] [192] Only Love (We Will Take)
analyzing Stamina RPG 1 Unlocks/[22] [192] Cranked Pastry Megamix (Full)
analyzing Stamina RPG 1 Unlocks/[16] [162] Nocturnal Nectar
analyzing Stamina RPG 1 Unlocks/[16] [140] TranceManiaX
analyzing Stamina RPG 1 Unlocks/[21] [240] Call of the Hound
analyzing Stamina RPG 1 Unlocks/[19] [175] NFG Talents Mix 001
analyzing Stamina RPG 1 Unlocks/[20] [174] Pendulum Essential Mix (Complete)
analyzing Stamina RPG 1 Unlocks/[23] [203] Stream-P

In [5]:
import pandas as pd

columns = [
    'name', 'pack', 'hash', 'meter',
    'bpm', 'totalStream', 'streamDensity',
    'minBpm', 'maxBpm',
    'segments', 'densityGraph']
df = pd.DataFrame(hash_to_data.values(), columns=columns)
if existing_df is not None:
    df = pd.concat([existing_df, df], axis=0, ignore_index=True)
df

,name,pack,hash,meter,bpm,totalStream,streamDensity,minBpm,maxBpm,segments,densityGraph
0,[21] [140] 140 BPM Stamina Killer,Stamina RPG 1 Unlocks,b9c0db86a2e1346db5967b712aeb1ff876154c28,21,140.000,8347.0,1.000000,140.000,140.000,[8347.0],"[[0.039, 0.0], [1.753, 9.333333333333332], [14..."
1,[20] [174] Twostament of Ian,Stamina RPG 1 Unlocks,c058d69d708e47f18dd6ddfbf92fbffc51994010,20,174.051,3276.0,0.994838,173.000,194.000,"[3242.0, -17.0, 34.0]","[[0.000, 11.533333333333333], [97.110, 11.5333..."
2,[26] [250] Chambers of Shaolin,Stamina RPG 4,d811758a6147f4376e1c9a919a7647bfddd31e8e,25,250.242,435.0,0.838150,242.243,250.268,"[272.0, -80.0, 140.0, -2.0, 16.0, -2.0, 7.0]","[[-0.009, 16.683933533333335], [259.882, 16.68..."
3,[21] [160] Eternal Nectar,Stamina RPG 1 Unlocks,4a9afccab24ddb0e70237271b19317a2a5b2f970,21,162.000,4900.0,1.000000,162.000,162.000,[4900.0],"[[24.845, 8.775000000000004], [26.326, 10.8000..."
4,[18] [174] Enjoy the Flight,Stamina RPG 1 Unlocks,c1929e78ef4d6908a6fa7ebfa9e71693c057a051,18,174.000,1246.0,0.665598,173.500,174.040,"[6.0, -3.0, 1.0, -3.0, 1.0, -3.0, 1.0, -3.0, 1...","[[0.002, 0.0], [77.243, 0.0], [78.623, 0.72499..."
...,...,...,...,...,...,...,...,...,...,...,...
4570,[29] [296] Oblivion,Stamina RPG 10 - FE,4406a6fec1b9411c35b48f531363c93ed0f94cb3,29,296.000,288.0,1.000000,296.000,296.000,[288.0],"[[0.051, 9.866666666666665], [0.862, 19.733333..."
4571,[29] [324] Somewhere I Belong,Stamina RPG 10 - FE,0441c9df5063812a6f2373aed44f762255bce2c0,29,324.066,169.0,0.665354,324.066,324.066,"[16.0, -40.0, 32.0, -32.0, 49.0, 1.0, 12.0, -8...","[[0.127, 0.0], [11.236, 0.0], [11.976, 4.05082..."
4572,[29] [318] XTREME FB 318,Stamina RPG 10 - FE,d1b28cdedc4c1df7a4c7885367a856e1da3c3597,29,318.000,192.0,0.836601,318.000,318.000,"[10.5, -1.5, 22.5, -10.5, 4.5, -1.5, 16.5, -13...","[[-0.009, 0.0], [9.048, 0.0], [10.180, 4.41666..."
4573,[30] [316] Kagayaku Hari no Kobito-zoku,Stamina RPG 10 - FE,a406b8532481a84a255c21771222ef7891cca54e,30,316.000,300.0,1.000000,316.000,316.000,[300.0],"[[0.000, 0.0], [0.759, 0.0], [1.519, 1.3166666..."


In [6]:
df.to_json('../public/dataset.json', orient='records', index=False)